<a href="https://colab.research.google.com/github/maxpopp8/rush-sportswear-sales-analysis/blob/main/GB885_Final_Project_Popp_M.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Final Project
You work as an analyst for RUSH, a globally renowned sportswear and footwear brand known for its innovative designs and performance-oriented products. The company stores its raw sales data as a collection of three tables:
* TABLE_PRODUCTS
* TABLE_RETAILER
* TABLE_SALES

The data includes the number of units sold, the total sales revenue, the location of the sales, the type of product sold, as well as other relevant information. (For data field definitions and explanations, see the data dictionary.) The data is “raw,” meaning it has not been cleaned and probably contains errors that need to be addressed.

The VP of US Sales has tasked you with analyzing sales data for trends and insights that will help company leadership understand the market and identify opportunities for growth. For example, you may want to look for trends or insights in seasonality, retailers, locations, or sales methods. Take the initiative to apply your creativity and curiosity to the data.

In addition, she has asked for you to query the data warehouse to answer the following business questions:
* What product category (product) had the highest sales (in dollars) in 2021
* How much did it sell?
* What state had the highest sales (in dollars) of women's products in 2021, and how much was it?
* What state had the highest sales (in dollars) of men's products in 2021, and how much was it?
* What retailer purchased the most units in 2021? In 2020?


In [ ]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the Data

In [ ]:
# load data into collab notebook
products = pd.read_csv('TABLE_PRODUCTS_885.csv', sep='|')
retailers = pd.read_csv('TABLE_RETAILER_885.csv')
sales = pd.read_csv('TABLE_SALES_885.csv')

# Data Inspection

In [ ]:
# inspect the tables

print('Products:')
print(products.shape)

print('n/Retailers:')
print(retailers.shape)

print('n/Sales:')
print(sales.shape)

In [ ]:
# data preview of products
display(products.head())

In [ ]:
# data preview of retailers
display(retailers.head())

In [ ]:
# data preview of sales
display(sales.head())

In [ ]:
# check datas columns
print('Products columns:')
print(products.columns.tolist())

print('\nRetailers columns:')
print(retailers.columns.tolist())

print('\nSales columns:')
print(sales.columns.tolist())

In [ ]:
# check data types of sales data
sales.info()

In [ ]:
# check data tables for missing values
print('Missing values in Products:')
print(products.isnull().sum())

print('\nMissing values in Retailers:')
print(retailers.isnull().sum())

print('\nMissing values in Sales:')
print(sales.isnull().sum())

In [ ]:
# check data tables for duplicates
print("Duplicate product rows:", products.duplicated().sum())

print("Duplicate retailer rows:", retailers.duplicated().sum())

print("Duplicate sales rows:", sales.duplicated().sum())

In [ ]:
# check units_sold and why pandas sees it as an onject
sales['UNITS_SOLD'].unique()

In [ ]:
# check units_sold and why pandas sees it as an onject
sales['UNITS_SOLD'].value_counts().head(20)

In [ ]:
# check the date value of the data
sales['INVOICE_DATE'].head(10)

In [ ]:
# check the date value of the data
sales['INVOICE_DATE'].unique()[:10]

In [ ]:
# check the sales method
sales['SALES_METHOD'].value_counts()

In [ ]:
# check the retailers IDs
print("Number of unique retailer IDs in sales:", sales['RETAILER_ID'].nunique())
print("Number of retailer IDs in retailer table:", retailers['RETAILER_ID'].nunique())

In [ ]:
# check the product IDs
print('Product IDs in sales:')
print(sorted(sales['PRODUCT_ID'].unique()))

print('\nProduct IDs in products table:')
print(sorted(products['PRODUCT_ID'].unique()))

In [ ]:
# investigate the units_sold problem
print(sales[~sales['UNITS_SOLD'].str.isnumeric()][['ORDER_ID', 'UNITS_SOLD']])

In [ ]:
# investigate the units_sold problem
print('Number of non-numeric units sold values:',
      (~sales['UNITS_SOLD'].str.isnumeric()).sum())

In [ ]:
# investigate the duplicate outlet
print(sales['SALES_METHOD'].unique())

In [ ]:
# investigate the duplicate outlet
for method in sales['SALES_METHOD'].unique():
    print(repr(method))

In [ ]:
# investigate the retailer ID that does not match
sales_retailer_ids = set(sales['RETAILER_ID'])
retailer_ids = set(retailers['RETAILER_ID'])

print('Retailer IDs in sales but not retailer table:')
print(sales_retailer_ids - retailer_ids)

In [ ]:
# locate missing prices
sales[sales['PRICE_PER_UNIT'].isnull()]

In [ ]:
# inspect rows with invalid units sold values
sales[sales['UNITS_SOLD'] == '***']

In [ ]:
# inspect rows around the invalid units sold records
sales.loc[[1010,1011,1012,1013,1014]]

In [ ]:
# inspect rows around the invalid units sold records
sales.loc[[1437,1438,1439,1440,1441]]

In [ ]:
# inspect the incorrect sales method value
sales[sales['SALES_METHOD'] == 'Ootlet']

In [ ]:
# inspect records using the invlaid retailer ID
sales[sales['RETAILER_ID'] == '999999999']

In [ ]:
# count records using the invalid retailer ID
(sales['RETAILER_ID'] == '999999999').sum()

In [ ]:
# inspect nearby sales for the same retailer and product
sales[(sales['RETAILER_ID'] == 'A00NVEBU') & (sales['PRODUCT_ID'] == 20)]

In [ ]:
# summarize sales records with the invlaid retailer ID
invalid_retailer = sales[sales['RETAILER_ID'] == '999999999']

print("Number of records:", len(invalid_retailer))
print("Total units:", invalid_retailer['UNITS_SOLD'].replace('***', np.nan).astype(float).sum())

# Cleaning the Data

In [ ]:
# creating a copy of the raw sales data for cleaning
sales_clean = sales.copy()

In [ ]:
# correct the misspelled saled method
sales_clean['SALES_METHOD'] = sales_clean['SALES_METHOD'].replace({'Ootlet': 'Outlet'})
sales_clean['SALES_METHOD'].value_counts()

In [ ]:
# convert units sold to numeric (*** becomes missing value)
sales_clean['UNITS_SOLD'] = pd.to_numeric(sales_clean['UNITS_SOLD'], errors='coerce')
sales_clean['UNITS_SOLD'].isnull().sum()

In [ ]:
# convert invoice data from text to datetime
sales_clean['INVOICE_DATE'] = pd.to_datetime(sales_clean['INVOICE_DATE'])
sales_clean['INVOICE_DATE'].dtype

In [ ]:
# making sure price per unit is numeric
sales_clean['PRICE_PER_UNIT'] = pd.to_numeric(sales_clean['PRICE_PER_UNIT'], errors='coerce')

In [ ]:
# calculate total sales dollars for each order
sales_clean['SALES_DOLLARS'] = (sales_clean['PRICE_PER_UNIT'] * sales_clean['UNITS_SOLD'])
sales_clean[['PRICE_PER_UNIT', 'UNITS_SOLD', 'SALES_DOLLARS']].head()

In [ ]:
# verify the cleaned sales data
print('Missing values:')
print(sales_clean.isnull().sum())

print('\nData types:')
print(sales_clean.dtypes)

print('\nSales methods:')
print(sales_clean['SALES_METHOD'].unique())

# Merge the Tables

In [ ]:
# merge product information into the sales data
sales_analysis = sales_clean.merge(products, on='PRODUCT_ID', how='left')

In [ ]:
# merge retailer information into the sales data
sales_analysis = sales_analysis.merge(retailers, on='RETAILER_ID', how='left')
sales_analysis.head()

In [ ]:
# check for sales records without matching retailer information
sales_analysis[sales_analysis['RETAILER_ID'].isnull()]

In [ ]:
# verify the merged data
print("Rows:", sales_analysis.shape[0])
print("Columns:", sales_analysis.shape[1])

In [ ]:
# check for duplicate retailer IDs
retailers[retailers['RETAILER_ID'].duplicated(keep=False)].sort_values('RETAILER_ID')

In [ ]:
# count the number of duplicate retailer IDs
retailers['RETAILER_ID'].value_counts()[retailers['RETAILER_ID'].value_counts() > 1]

In [ ]:
# check products table to make sure merge isn't causing issues
print('Duplicate Product IDs:')
print(products['PRODUCT_ID'].duplicated().sum())

print('Duplicate Retailer IDs:')
print(retailers['RETAILER_ID'].duplicated().sum())

In [ ]:
# remerge data
sales_analysis = sales_clean.merge(products, on='PRODUCT_ID', how='left').merge(retailers, on='RETAILER_ID', how='left')

In [ ]:
# verify the merged data
print("Rows:", sales_analysis.shape[0])
print("Columns:", sales_analysis.shape[1])

In [ ]:
# check for duplicate retailer IDs
duplicate_retailers = retailers[retailers['RETAILER_ID'].duplicated(keep=False)].sort_values('RETAILER_ID')

display(duplicate_retailers)

In [ ]:
# count how many times each retailer ID appears
retailer_id_counts = retailers['RETAILER_ID'].value_counts()

print(retailer_id_counts[retailer_id_counts > 1])

In [ ]:
# check whether products IDs have duplicates
print('Duplicate product IDs:', products['PRODUCT_ID'].duplicated().sum())

print('Duplicate retailer IDs:', retailers['RETAILER_ID'].duplicated().sum())

In [ ]:
# check sales associated with duplicate retailer IDs
duplicate_ids = retailers[retailers['RETAILER_ID'].duplicated(keep=False)]['RETAILER_ID'].unique()

sales[sales['RETAILER_ID'].isin(duplicate_ids)]['RETAILER_ID'].value_counts()

In [ ]:
# summarize sales for the duplicate retailer IDs
sales[sales['RETAILER_ID'].isin(duplicate_ids)].groupby('RETAILER_ID').agg(records=('ORDER_ID', 'count'), units=('UNITS_SOLD', lambda x: pd.to_numeric(x, errors='coerce').sum()), sales_dollars=('PRICE_PER_UNIT', lambda x: x.sum()))

In [ ]:
# check duplicate retailer IDs by year
sales[sales['RETAILER_ID'].isin(duplicate_ids)].groupby(['RETAILER_ID', 'YEAR']).size()

In [ ]:
# merge sales with product information (product_id is unique in both tables)
sales_analysis = sales_clean.merge(products, on='PRODUCT_ID', how='left')

print("Rows:", sales_analysis.shape[0])

In [ ]:
# indentify retailer IDs that appear only once in the retailer table
valid_retailer_ids = retailers[~retailers['RETAILER_ID'].duplicated(keep=False)]['RETAILER_ID']

In [ ]:
# keep only sales with an unambious retailer ID
sales_location = sales_analysis[sales_analysis['RETAILER_ID'].isin(valid_retailer_ids)].merge(retailers, on='RETAILER_ID', how='left')

print('Rows available for location analysis:', sales_location.shape[0])

In [ ]:
# verify that the location merge did not duplicate rows
print('Original sales rows:', len(sales))
print('Product analysis rows:', len(sales_analysis))
print('Location analysis rows:', len(sales_location))

# VP Business Questions

In [ ]:
# 1: find the product with the highest sales dollars in 2021
product_sales_2021 = (sales_analysis[sales_analysis['YEAR'] == 2021].groupby('PRODUCT_NAME')['SALES_DOLLARS'].sum().sort_values(ascending=False))

display(product_sales_2021)

In [ ]:
# 1: display the top selling product
print('Top-selling product in 2021:', product_sales_2021.index[0])
print('2021 sales: ${:,.0f}'.format(product_sales_2021.iloc[0]))

In [ ]:
# 2: find the data with the highest sales of womens products in 2021
women_sales_2021 = (sales_location[(sales_location['YEAR'] == 2021) &(sales_location['PRODUCT_NAME'].str.contains("Women's"))].groupby('STATE')['SALES_DOLLARS'].sum().sort_values(ascending=False))

display(women_sales_2021.head(10))

In [ ]:
# 2: display the state with the highest womens product sales
print('Top state for womens products:', women_sales_2021.index[0])
print('2021 sales: ${:,.0f}'.format(women_sales_2021.iloc[0]))

In [ ]:
# 3: find the state with the highest sales of mens products in 2021
men_sales_2021 = (sales_location[(sales_location['YEAR'] == 2021) &(sales_location['PRODUCT_NAME'].str.contains("Men's"))].groupby('STATE')['SALES_DOLLARS'].sum().sort_values(ascending=False))

display(men_sales_2021.head(10))

In [ ]:
# 3: display the state with the highest mens product sales
print('Top state for mens products:', men_sales_2021.index[0])
print('2021 sales: ${:,.0f}'.format(men_sales_2021.iloc[0]))

In [ ]:
# 4: find the retailer with the most units sold in 2021 and 2020
retailer_units = (sales_location[sales_location['YEAR'].isin([2020, 2021])].groupby(['YEAR', 'RETAILER'])['UNITS_SOLD'].sum().reset_index())

retailer_units['RANK'] = (retailer_units.groupby('YEAR')['UNITS_SOLD'].rank(method='dense', ascending=False))

display(retailer_units[retailer_units['RANK'] == 1])

# Business Insights

In [ ]:
# 2021 sales by product

product_sales = (sales_analysis[sales_analysis['YEAR'] == 2021].groupby('PRODUCT_NAME')['SALES_DOLLARS'].sum().sort_values(ascending=False))

display(product_sales)

In [ ]:
# visualize 2021 sales by product
product_sales.plot(kind='barh')

plt.title('RUSH Sales by Product - 2021')
plt.xlabel('Sales ($)')
plt.ylabel('Product')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# 2021 sales by sales method
method_sales = (sales_analysis[sales_analysis['YEAR'] == 2021].groupby('SALES_METHOD')['SALES_DOLLARS'].sum().sort_values(ascending=False))

display(method_sales)

In [ ]:
# visualize sales by sales method
method_sales.plot(kind='bar')

plt.title('RUSH Sales by Sales Method - 2021')
plt.xlabel('Sales Method')
plt.ylabel('Sales ($)')
plt.xticks(rotation=0)
plt.show()

In [ ]:
# calculate the percentage of 2021 sales by sales method
method_percentage = (method_sales / method_sales.sum() * 100).round(1)

display(method_percentage)

In [ ]:
# calculate monthly sales for 2021
monthly_sales = (sales_analysis[sales_analysis['YEAR'] == 2021].groupby('MONTH')['SALES_DOLLARS'].sum())

display(monthly_sales)

In [ ]:
# visualize monthly sales trend
monthly_sales.plot(kind='line', marker='o')

plt.title('Monthly RUSH Sales - 2021')
plt.xlabel('Month')
plt.ylabel('Sales ($)')
plt.xticks(range(1, 13))
plt.grid()
plt.show()

In [ ]:
# 2021 sales by region
region_sales = (sales_location[sales_location['YEAR'] == 2021].groupby('REGION')['SALES_DOLLARS'].sum().sort_values(ascending=False))

display(region_sales)

In [ ]:
# visualize sales by region
region_sales.plot(kind='bar')

plt.title('RUSH Sales by Region - 2021')
plt.xlabel('Region')
plt.ylabel('Sales ($)')
plt.xticks(rotation=0)
plt.show()

In [ ]:
# percentage of 2021 sales by region
region_percentage = (region_sales / region_sales.sum() * 100).round(1)

display(region_percentage)

In [ ]:
# average operating margin by product in 2021
product_margin = (sales_analysis[sales_analysis['YEAR'] == 2021].groupby('PRODUCT_NAME')['OPERATING_MARGIN'].mean().sort_values(ascending=False))

display(product_margin)

In [ ]:
# visualize average operating margin by product
product_margin.plot(kind='barh')

plt.title('Average Operating Margin by Product - 2021')
plt.xlabel('Average Operating Margin')
plt.ylabel('Product')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# investigate Mays sales spike
may_sales = (sales_analysis[(sales_analysis['YEAR'] == 2021) &(sales_analysis['MONTH'] == 5)].groupby('PRODUCT_NAME')['SALES_DOLLARS'].sum().sort_values(ascending=False))

display(may_sales)

In [ ]:
# identify the largest individual sales records in May
may_orders = sales_analysis[(sales_analysis['YEAR'] == 2021) &(sales_analysis['MONTH'] == 5)].sort_values('SALES_DOLLARS', ascending=False)

display(may_orders[['ORDER_ID', 'PRODUCT_NAME', 'PRICE_PER_UNIT','UNITS_SOLD', 'SALES_DOLLARS', 'SALES_METHOD']].head(10))

In [ ]:
# investigate unusually large unit quantities in May
display(may_orders[['ORDER_ID', 'PRODUCT_NAME', 'UNITS_SOLD', 'PRICE_PER_UNIT', 'SALES_DOLLARS']].head(20))